# VR experiment: corrected SSQ analysis

This executable replacement focuses on the analyses affected by the Total SSQ scoring error. It uses the restricted pseudonymized dataset, never participant names. The corrected formula is `Total SSQ = (N/9.54 + O/7.58 + D/13.92) * 3.74`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_PATH = Path('restricted/vr_experiment_internal_pseudonymized.csv')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} trials from {df.participant_id.nunique()} participants.')
df.head()

## Validate corrected scoring

The subscale columns are already weighted. We recover their integer raw sums before applying the Total SSQ multiplier.

In [ ]:
recalculated = (
    df['nausea_subscore'] / 9.54
    + df['oculomotor_subscore'] / 7.58
    + df['disorientation_subscore'] / 13.92
) * 3.74
assert np.allclose(recalculated, df['ssq_total_score'], atol=0.01)
assert np.allclose(df['nausea_subscore'] / 9.54, df['nausea_raw_sum'])
assert np.allclose(df['oculomotor_subscore'] / 7.58, df['oculomotor_raw_sum'])
assert np.allclose(df['disorientation_subscore'] / 13.92, df['disorientation_raw_sum'])
print('All 76 corrected totals and raw subscale sums validated.')

## Old versus corrected totals

In [ ]:
comparison = pd.DataFrame({
    'legacy_incorrect': df['legacy_total_ssq_incorrect'],
    'corrected': df['ssq_total_score'],
}).agg(['count', 'mean', 'std', 'median', 'min', 'max']).round(2)
comparison.loc['correlation', 'legacy_incorrect'] = df['legacy_total_ssq_incorrect'].corr(df['ssq_total_score'])
comparison

## Corrected descriptive statistics by experimental condition

In [ ]:
condition_summary = (
    df.groupby(['fps', 'posture'])['ssq_total_score']
      .agg(n='count', mean='mean', std='std', median='median', min='min', max='max')
      .round(2)
)
condition_summary

## Within-participant effect checks

These paired checks respect the repeated-measures design. They are a compact diagnostic, not a substitute for the final preregistered or reviewer-approved model.

In [ ]:
wide = df.pivot(index='participant_id', columns=['fps', 'posture'], values='ssq_total_score')
fps_difference = wide[90].mean(axis=1) - wide[60].mean(axis=1)
posture_difference = wide.xs('Standing', level='posture', axis=1).mean(axis=1) - wide.xs('Sitting', level='posture', axis=1).mean(axis=1)
interaction_difference = (wide[(90, 'Standing')] - wide[(90, 'Sitting')]) - (wide[(60, 'Standing')] - wide[(60, 'Sitting')])

def effect_row(label, values):
    test = stats.ttest_1samp(values, 0.0)
    return {'contrast': label, 'mean_difference': values.mean(), 'sd_difference': values.std(ddof=1), 't': test.statistic, 'p': test.pvalue}

paired_results = pd.DataFrame([
    effect_row('90 FPS minus 60 FPS', fps_difference),
    effect_row('Standing minus Sitting', posture_difference),
    effect_row('FPS × posture difference-in-differences', interaction_difference),
]).set_index('contrast').round(4)
paired_results

In [ ]:
plot_order = [(60, 'Sitting'), (60, 'Standing'), (90, 'Sitting'), (90, 'Standing')]
labels = [f'{fps} FPS\n{posture}' for fps, posture in plot_order]
means = [df.loc[(df.fps == fps) & (df.posture == posture), 'ssq_total_score'].mean() for fps, posture in plot_order]
errors = [df.loc[(df.fps == fps) & (df.posture == posture), 'ssq_total_score'].sem() for fps, posture in plot_order]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(labels, means, yerr=errors, capsize=5, color=['#4C78A8', '#72B7B2', '#F58518', '#E45756'])
ax.set_ylabel('Corrected Total SSQ score')
ax.set_title('Corrected Total SSQ by VR condition (mean ± SEM)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## What must be updated

The original notebook must not continue using `Post-VR SSQ Total Score at 60 FPS`, `Post-VR SSQ Total Score at 90 FPS`, or `Pre-VR SSQ Total Score` as originally stored. Any plot, model, effect size, threshold, or prose conclusion based on those values must be regenerated. The N/O/D weighted subscales remain usable if their symptom severities were summed correctly. After regenerating results, update the corresponding numbers, figures, tables, methods, and conclusions in Overleaf.